## 1. Download data from Don’tGetKicked competition. <br/><br/>
## 2. Design the train/validation/test split.
Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set.<br/><br/>


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [172]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, precision_recall_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer

In [173]:
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ML4_Classification_problems.ID_1254802-1/src/data/training.csv", parse_dates=["PurchDate"])
train_df

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,2009-12-02,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,2009-12-02,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,2009-12-02,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,2009-12-02,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [174]:
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ML4_Classification_problems.ID_1254802-1/src/data/test.csv", parse_dates=["PurchDate"])
test_df

,RefId,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,73015,2009-12-02,ADESA,2005,4,PONTIAC,GRAND PRIX,Bas,4D SEDAN,SILVER,...,8557.0,9752.0,NaN,NaN,18881,30212,GA,6500.0,0,2152
1,73016,2009-12-02,ADESA,2005,4,CHEVROLET,MALIBU V6,LS,4D SEDAN LS,SILVER,...,7562.0,9296.0,NaN,NaN,18111,30212,GA,6300.0,0,1118
2,73017,2009-12-02,ADESA,2006,3,DODGE,DURANGO 2WD V8,Adv,4D SUV 4.7L ADVENTURER,SILVER,...,15340.0,16512.0,NaN,NaN,18111,30212,GA,9700.0,0,1215
3,73018,2009-12-02,ADESA,2002,7,SATURN,L SERIES,L20,4D SEDAN L200,GOLD,...,5725.0,6398.0,NaN,NaN,18881,30212,GA,4150.0,0,1933
4,73019,2009-12-02,ADESA,2007,2,HYUNDAI,ACCENT,GS,2D COUPE GS,BLUE,...,5914.0,7350.0,NaN,NaN,18111,30212,GA,4100.0,0,920
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48702,121742,2010-11-17,MANHEIM,2005,5,FORD,FIVE HUNDRED,SEL,4D SEDAN SEL,BLACK,...,9764.0,11395.0,NaN,NaN,20928,33411,FL,7955.0,0,1633
48703,121743,2010-11-17,MANHEIM,2007,3,TOYOTA,COROLLA,CE,4D SEDAN CE,GREEN,...,10283.0,11565.0,NaN,NaN,20928,33411,FL,7035.0,0,594
48704,121744,2010-11-17,MANHEIM,2006,4,KIA,SPECTRA,EX,4D SEDAN EX,BLACK,...,7871.0,9490.0,NO,GREEN,20928,33411,FL,6335.0,0,594
48705,121745,2010-11-17,MANHEIM,2005,5,MAZDA,MAZDA3,s,4D SEDAN GT,SILVER,...,8576.0,9937.0,NO,GREEN,20928,33411,FL,8055.0,0,1038


In [175]:
X_train_df = train_df.drop("IsBadBuy", axis=1).copy()
y_train_df = train_df["IsBadBuy"].copy()

In [176]:
def split_into_3(X, y):
    df = pd.concat([X, y], axis=1)
    df = df.sort_values(by="PurchDate")
    ind = df.index
    n_samples = len(ind)
    train_count = n_samples // 3
    validation_count = 2 * n_samples // 3

    train_ind = ind[:train_count]
    validation_ind = ind[train_count:validation_count]
    test_ind = ind[validation_count:]

    return (X.loc[train_ind], X.loc[validation_ind], X.loc[test_ind],
            y.loc[train_ind], y.loc[validation_ind], y.loc[test_ind])


In [177]:
X_train, X_valid, X_test, y_train, y_valid, y_test = split_into_3(
    X_train_df, y_train_df
)

X_train.shape, X_valid.shape, X_test.shape, y_train.shape, y_valid.shape, y_test.shape

((24327, 33), (24328, 33), (24328, 33), (24327,), (24328,), (24328,))

## 3. Use LabelEncoder or OneHotEncoder
From sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html <br/><br/>

In [178]:
X_train_encoded = X_train.copy()
X_valid_encoded = X_valid.copy()
X_test_encoded = X_test.copy()

categorical_cols = X_train_encoded.select_dtypes(include=["object"]).columns
numerical_cols = X_train_encoded.select_dtypes(include=[np.number]).columns

le = LabelEncoder()

for col in categorical_cols:
    le.fit(train_df[col].astype(str))
    X_train_encoded[col] = le.transform(X_train_encoded[col].astype(str))
    X_valid_encoded[col] = le.transform(X_valid_encoded[col].astype(str))
    X_test_encoded[col] = le.transform(X_test_encoded[col].astype(str))

## 4. Train LogisticRegression, GaussianNB, KNN
From sklearn on the training dataset and check the quality of your algorithms on the validation dataset. The dependent variable (IsBadBuy) is binary. Don't forget to normalize your datasets before training your models.

You must get at least 0.15 Gini score (the best of all three). Which algorithm performs better? And why?

In [179]:
for df in [X_train_encoded, X_valid_encoded, X_test_encoded]:
    if "PurchDate" in df.columns:
        df["PurchYear"] = df["PurchDate"].dt.year
        df["PurchMonth"] = df["PurchDate"].dt.month
        df["PurchDay"] = df["PurchDate"].dt.day
        df.drop(columns=["PurchDate"], inplace=True)

num_means = X_train_encoded[numerical_cols].mean()
for df in [X_train_encoded, X_valid_encoded, X_test_encoded]:
    df[numerical_cols] = df[numerical_cols].fillna(num_means)

scaler = StandardScaler()
X_train_encoded[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])
X_valid_encoded[numerical_cols] = scaler.transform(X_valid_encoded[numerical_cols])
X_test_encoded[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

In [180]:
result = {}

In [181]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(X_train_encoded, y_train)

y_pred_lr = lr.predict(X_valid_encoded)

In [182]:
auc = roc_auc_score(
    y_valid.to_numpy(),
    y_pred_lr
)
gini = abs(2 * auc - 1)
result["Lr_gini"] = gini
gini

np.float64(0.20269386198589734)

In [183]:
gnb = GaussianNB()
gnb.fit(X_train_encoded, y_train)
y_pred_gnb = gnb.predict(X_valid_encoded)

In [184]:
auc = roc_auc_score(
    y_valid.to_numpy(),
    y_pred_gnb
)
gini = abs(2 * auc - 1)
result["GNB_gini"] = gini
gini

np.float64(0.1781446250472798)

In [185]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_encoded, y_train)
y_pred_knn = knn_model.predict(X_valid_encoded)

In [186]:
auc = roc_auc_score(
    y_valid.to_numpy(),
    y_pred_knn
)
result["Knn_gini"] = gini
gini = abs(2 * auc - 1)
gini

np.float64(0.019537147855732018)

## 5. Implement Gini score calculation.
You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AUC calculation. Check if your metric is approximately equal to abs(2*sklearn.metrics.roc_auc_score - 1).

In [187]:
def roc_auc(y, y_pred):
  y = np.array(y)
  y_pred = np.array(y_pred)
  positive, negative = np.where(y == 1)[0], np.where(y == 0)[0]
  diff_matrix = y_pred[positive, None] - y_pred[negative]
  count_positive = np.sum(diff_matrix > 0) + 0.5 * np.sum(diff_matrix == 0)

  auc = count_positive / (len(positive) * len(negative))

  return auc

def gini_fun(y, y_pred):
    auc = roc_auc(y, y_pred)
    gini = abs(2 * auc - 1)
    return gini

In [188]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result["Lr_gini_my"] = gini
gini

np.float64(0.20269386198589734)

In [189]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_gnb
)
result["GNB_gini_my"] = gini
gini

np.float64(0.17814462504728001)

In [190]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_knn
)
result["Knn_gini_my"] = gini
gini

np.float64(0.019537147855732018)

## 6. Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers.
For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?

In [191]:
class logistic_regression():
  def __init__(self, lr_speed = 0.1, max_iter = 500):
    self.lr_speed = lr_speed
    self.max_iter = max_iter
    self.weights = None
    self.bias = None
    self.scaler = StandardScaler()

  def fit(self, X, y):
    X = np.array(X)
    y = np.array(y)
    self.bias = 0

    X = self.scaler.fit_transform(X)

    n_samples, n_features = X.shape
    self.weights = np.zeros(n_features)
    np.random.seed(21)

    for _ in range(self.max_iter):
      indices = np.random.permutation(n_samples)

      xi = X[indices]
      yi = y[indices]
      z = np.dot(xi, self.weights) + self.bias
      z = np.clip(z, -500, 500)
      y_pred = 1 / (1 + np.exp(-z))

      error = yi - y_pred

      self.weights += self.lr_speed * np.dot((error * y_pred * (1 - y_pred)), xi)
      self.bias += self.lr_speed * (error * y_pred * (1 - y_pred)).sum()

  def predict(self, X):
      X = np.array(X)
      X = self.scaler.transform(X)
      linear_model = np.dot(X, self.weights) + self.bias
      linear_model = np.clip(linear_model, -500, 500)
      prob = 1 / (1 + np.exp(-linear_model))
      return prob

  def predict_proba(self, X):
      prob = self.predict(X)
      return np.column_stack([1 - prob, prob])

In [192]:
lr_my = logistic_regression(max_iter=1000)
lr_my.fit(X_train_encoded, y_train)

y_pred_proba_lr = lr_my.predict_proba(X_valid_encoded)
y_pred_lr = lr_my.predict(X_valid_encoded)

In [193]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_proba_lr[:, 1]
)
gini

np.float64(0.37620492100580605)

In [194]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result["Lr_my_gini_my"] = gini
gini

np.float64(0.37620492100580605)

In [195]:
class knn ():
  def __init__(self, n_neighbors=5):
    self.n_neighbors = n_neighbors
    self.neighbors_classes = None

  def fit(self, X, y):
    self.X_train = np.array(X)
    self.y_train = np.array(y)
    self.classes_ = np.unique(y)

  def predict(self, X):
    proba = self.predict_proba(X)
    class_indices = np.argmax(proba, axis=1)
    return self.classes_[class_indices]

  def predict_proba(self, X):
    X = np.array(X)
    predictions = []

    for xi in X:
      distances = np.sqrt(np.sum((self.X_train - xi) ** 2, axis=1))
      neighbors_idx = distances.argsort()[:self.n_neighbors]
      neighbors_classes = self.y_train[neighbors_idx]

      proba = []
      for cls in self.classes_:
        proba.append(np.mean(neighbors_classes == cls))
      predictions.append(proba)

    return np.array(predictions)

In [196]:
knn_my = knn(n_neighbors=3)
knn_my.fit(X_train_encoded, y_train)
y_pred_proba_knn = knn_my.predict_proba(X_valid_encoded)
y_pred_knn = knn_my.predict(X_valid_encoded)

In [197]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_knn
)
result["Knn_my_gini_my"] = gini
gini

np.float64(0.019537147855732018)

In [198]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_proba_knn[:, 1]
)
gini

np.float64(0.07521849114769474)

In [199]:
class gaussian_nb:
  def __init__(self):
    self.classes_ = None
    self.Pc_ = None
    self.uci_ = None
    self.oci_ = None

  def fit(self, X, y):
    X = np.array(X)
    y = np.array(y)
    self.classes_ = np.unique(y)

    self.Pc_ = {}
    self.uci_ = {}
    self.oci_ = {}

    for cls in self.classes_:
      X_c = X[y == cls]
      Nc = X_c.shape[0]

      self.Pc_[cls] = Nc / X.shape[0]
      self.uci_[cls] = X_c.mean(axis=0)
      self.oci_[cls] = X_c.var(axis=0)
      self.oci_[cls][self.oci_[cls] == 0] = 1e-9

  def predict_proba(self, X):
    X = np.array(X)
    predictions = []

    for x in X:
      logs_probs = []
      for cls in self.classes_:
        prob_log = -0.5 * np.log(2 * np.pi * self.oci_[cls]) - ((x - self.uci_[cls]) ** 2) / ((2 * self.oci_[cls]))
        prob = np.sum(prob_log)
        logs_probs.append(np.log(self.Pc_[cls]) + prob)


      max_log = max(logs_probs)
      exp_probs = [np.exp(lp - max_log) for lp in logs_probs]
      sum_exp = sum(exp_probs)
      normalized_probs = [p / sum_exp for p in exp_probs]

      predictions.append(normalized_probs)

    return np.array(predictions)

  def predict(self, X):
      proba = self.predict_proba(X)
      class_indices = np.argmax(proba, axis=1)
      return np.array([self.classes_[i] for i in class_indices])


In [200]:
gnb_my = gaussian_nb()
gnb_my.fit(X_train_encoded, y_train)

y_pred_proba_gnb = gnb_my.predict_proba(X_valid_encoded)
y_pred_gnb = gnb_my.predict(X_valid_encoded)

In [201]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_proba_gnb[:, 1]
)
gini

np.float64(0.3983037561577385)

In [202]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_gnb
)
result["GNB_my_gini_my"] = gini
gini

np.float64(0.0036362226627713046)

## 7. Try to create non-linear features, for example:

fractions: feature1/feature2
<br/>groupby features: `df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())`

Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?
<br/><br/>

In [203]:
categorical_cols

Index(['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT',
       'AUCGUART', 'VNST'],
      dtype='object')

In [204]:
numerical_cols

Index(['RefId', 'VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO',
       'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

In [205]:
train_df_features = X_train_encoded.copy()
valid_df_features = X_valid_encoded.copy()

num_cols = ['MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice']

for col in num_cols:
    mean_values = train_df_features.groupby('Model')[col].mean()
    new_col_name = f"Model_Mean_{col}"
    train_df_features[new_col_name] = train_df_features['Model'].map(mean_values)

    mean_values_valid = valid_df_features.groupby('Model')[col].mean()
    valid_df_features[new_col_name] = valid_df_features['Model'].map(mean_values_valid)

scaler = StandardScaler()
feat_cols = [f"Model_Mean_{col}" for col in num_cols]

train_df_features[feat_cols] = scaler.fit_transform(train_df_features[feat_cols])
valid_df_features[feat_cols] = scaler.transform(valid_df_features[feat_cols])


In [206]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(train_df_features, y_train)

y_pred_lr = lr.predict(valid_df_features)

In [207]:
gnb = GaussianNB()
gnb.fit(train_df_features, y_train)
y_pred_gnb = gnb.predict(valid_df_features)

In [208]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(train_df_features, y_train)
y_pred_knn = knn_model.predict(valid_df_features)

In [209]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result["Lr_gini_my_new_features"] = gini

gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_gnb
)
result["GNB_gini_my_new_features"] = gini

gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_knn
)
result["Knn_gini_my_new_features"] = gini

In [210]:
for key, value in result.items():
  print(f"{key}: {value}")

Lr_gini: 0.20269386198589734
GNB_gini: 0.1781446250472798
Knn_gini: 0.1781446250472798
Lr_gini_my: 0.20269386198589734
GNB_gini_my: 0.17814462504728001
Knn_gini_my: 0.019537147855732018
Lr_my_gini_my: 0.37620492100580605
Knn_my_gini_my: 0.019537147855732018
GNB_my_gini_my: 0.0036362226627713046
Lr_gini_my_new_features: 0.2031695283907673
GNB_gini_my_new_features: 0.1699902584858337
Knn_gini_my_new_features: 0.019203231592612058


## 8. Determine the best features for the problem using the coefficients of the logistic model.
Try to eliminate useless features by hand and by L1 regularization. Which approach is better in terms of Gini score?
<br/><br/>

In [211]:
best_features = dict(sorted(zip(X_train.columns, abs(lr.coef_[0]))))
for key, value in best_features.items():
  print(f"{key}: {value}")

AUCGUART: 0.2669714077719719
Auction: 0.22372649539191794
BYRNO: 0.08932946950311599
Color: 0.05381533892763111
IsOnlineSale: 0.01884915319622471
MMRAcquisitionAuctionAveragePrice: 0.03323450901762872
MMRAcquisitionAuctionCleanPrice: 0.016517993833282972
MMRAcquisitionRetailAveragePrice: 0.03159718713038363
MMRAcquisitonRetailCleanPrice: 0.009492292297402283
MMRCurrentAuctionAveragePrice: 0.01753746941382835
MMRCurrentAuctionCleanPrice: 0.04870459520942316
MMRCurrentRetailAveragePrice: 0.03655148172063122
MMRCurrentRetailCleanPrice: 0.004412496828550894
Make: 2.2910557723100527e-05
Model: 0.0015522526677362542
Nationality: 0.008321582435055229
PRIMEUNIT: 0.008823386313975995
PurchDate: 0.06879791947539929
RefId: 0.03850201961910432
Size: 0.0724682894338317
SubModel: 0.0018742942380952545
TopThreeAmericanName: 0.01781861472847356
Transmission: 0.6304618768741828
Trim: 0.00040725141549362304
VNST: 0.1771405778071058
VNZIP1: 0.011548167424641771
VehBCost: 0.0
VehOdo: 0.05670910846107887
V

In [212]:
best_10 = list(best_features.keys())[-10:]

for key in best_10:
    print(key)


Trim
VNST
VNZIP1
VehBCost
VehOdo
VehYear
VehicleAge
WarrantyCost
WheelType
WheelTypeID


In [213]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(train_df_features[best_10], y_train)

y_pred_lr = lr.predict(valid_df_features[best_10])

In [214]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result["Lr_gini_my_best_features_handle"] = gini
gini

np.float64(0.20307502829626722)

In [215]:
lr = LogisticRegression(max_iter=1000, random_state=42, penalty="l1", solver="liblinear")
lr.fit(train_df_features, y_train)

y_pred_lr = lr.predict(valid_df_features)

In [216]:
best_features = dict(sorted(zip(X_train.columns, abs(lr.coef_[0]))))
for key, value in best_features.items():
  print(f"{key}: {value}")

best_10 = list(best_features.keys())[-10:]

for key in best_10:
    print(key)

AUCGUART: 0.269581113884442
Auction: 0.44604822408203915
BYRNO: 0.08893628104586337
Color: 0.3789703393704606
IsOnlineSale: 0.0388430891708559
MMRAcquisitionAuctionAveragePrice: 0.327914324251679
MMRAcquisitionAuctionCleanPrice: 0.3010467434117966
MMRAcquisitionRetailAveragePrice: 0.24223191447335954
MMRAcquisitonRetailCleanPrice: 0.2584795628497395
MMRCurrentAuctionAveragePrice: 0.02014353859874583
MMRCurrentAuctionCleanPrice: 0.0
MMRCurrentRetailAveragePrice: 0.29145725182358057
MMRCurrentRetailCleanPrice: 0.31602191853584405
Make: 8.602245495986802e-05
Model: 0.0011910172889042258
Nationality: 0.006996662646911949
PRIMEUNIT: 1.3551122059228842
PurchDate: 0.048235851102335664
RefId: 0.06880660989741916
Size: 0.07208165222186749
SubModel: 0.004426542070448092
TopThreeAmericanName: 0.35720414336214346
Transmission: 0.6799610112238451
Trim: 0.0004944029472918718
VNST: 0.2767845115726807
VNZIP1: 0.010049293828137197
VehBCost: 0.0
VehOdo: 0.102029777654875
VehYear: 0.0
VehicleAge: 0.00011

In [217]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(train_df_features[best_10], y_train)

y_pred_lr = lr.predict(valid_df_features[best_10])

In [218]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result["Lr_gini_my_best_features_l1"] = gini
gini

np.float64(0.20307502829626722)

## 9. Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset.
Which hyperparameters have the most impact?
<br/><br/>

In [219]:
gnb = GaussianNB()
gnb.fit(train_df_features[best_10], y_train)
y_pred_gnb = gnb.predict(valid_df_features[best_10])

In [220]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(train_df_features[best_10], y_train)
y_pred_knn = knn_model.predict(valid_df_features[best_10])

In [221]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_gnb
)
result["GNB_gini_my_best_features"] = gini

gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_knn
)
result["Knn_gini_my_best_features"] = gini

In [222]:
for key, value in result.items():
  print(f"{key}: {value}")

Lr_gini: 0.20269386198589734
GNB_gini: 0.1781446250472798
Knn_gini: 0.1781446250472798
Lr_gini_my: 0.20269386198589734
GNB_gini_my: 0.17814462504728001
Knn_gini_my: 0.019537147855732018
Lr_my_gini_my: 0.37620492100580605
Knn_my_gini_my: 0.019537147855732018
GNB_my_gini_my: 0.0036362226627713046
Lr_gini_my_new_features: 0.2031695283907673
GNB_gini_my_new_features: 0.1699902584858337
Knn_gini_my_new_features: 0.019203231592612058
Lr_gini_my_best_features_handle: 0.20307502829626722
Lr_gini_my_best_features_l1: 0.20307502829626722
GNB_gini_my_best_features: 0.2678332191606527
Knn_gini_my_best_features: 0.1649647888585941


Лучшая модель - LogisticRegression с L1 регулиризацией и лучшими параметрами

In [223]:
best_features = dict(sorted(zip(X_train.columns, abs(lr.coef_[0]))))
for key, value in best_features.items():
  print(f"{key}: {value}")

Auction: 0.08613406789633717
Color: 0.6924595272884501
Make: 0.2093392170244824
Model: 0.2072200871505578
PurchDate: 0.009671062956755141
RefId: 0.0005259310798989476
SubModel: 1.303928362608885
Trim: 0.00683743380737749
VehYear: 0.1940426575837881
VehicleAge: 0.1903627780404801


## 10. Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini.
Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitted or not? Explain.
<br/><br/>

In [224]:
X_train, X_valid, X_test, y_train, y_valid, y_test = split_into_3(
    X_train_df, y_train_df
)

X_train.shape, X_valid.shape, X_test.shape, y_train.shape, y_valid.shape, y_test.shape

((24327, 33), (24328, 33), (24328, 33), (24327,), (24328,), (24328,))

In [225]:
X_train_encoded = X_train.copy()
X_valid_encoded = X_valid.copy()
X_test_encoded = X_test.copy()

for col in categorical_cols:
    le.fit(train_df[col].astype(str))
    X_train_encoded[col] = le.transform(X_train_encoded[col].astype(str))
    X_valid_encoded[col] = le.transform(X_valid_encoded[col].astype(str))
    X_test_encoded[col] = le.transform(X_test_encoded[col].astype(str))

for df in [X_train_encoded, X_valid_encoded, X_test_encoded]:
    if "PurchDate" in df.columns:
        df["PurchYear"] = df["PurchDate"].dt.year
        df["PurchMonth"] = df["PurchDate"].dt.month
        df["PurchDay"] = df["PurchDate"].dt.day
        df.drop(columns=["PurchDate"], inplace=True)

num_means = X_train_encoded[numerical_cols].mean()
for df in [X_train_encoded, X_valid_encoded, X_test_encoded]:
    df[numerical_cols] = df[numerical_cols].fillna(num_means)

scaler = StandardScaler()
X_train_encoded[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])
X_valid_encoded[numerical_cols] = scaler.transform(X_valid_encoded[numerical_cols])
X_test_encoded[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

In [226]:
lr = LogisticRegression(max_iter=1000, random_state=42, penalty="l1", solver="liblinear")
lr.fit(X_train_encoded[best_10], y_train)
y_pred_lr = lr.predict(X_test_encoded[best_10])

In [227]:
gnb = GaussianNB()
gnb.fit(X_train_encoded[best_10], y_train)
y_pred_gnb = gnb.predict(X_test_encoded[best_10])

In [228]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_encoded[best_10], y_train)
y_pred_knn = knn_model.predict(X_test_encoded[best_10])

In [229]:
result_for_test = {}

In [230]:
gini = gini_fun(
    y_test.to_numpy(),
    y_pred_lr
)
result_for_test["Lr_gini_my_test"] = gini

gini = gini_fun(
    y_test.to_numpy(),
    y_pred_gnb
)
result_for_test["GNB_gini_my_test"] = gini

gini = gini_fun(
    y_test.to_numpy(),
    y_pred_knn
)
result_for_test["Knn_gini_my_test"] = gini

In [231]:
lr = LogisticRegression(max_iter=1000, random_state=42, penalty="l1", solver="liblinear")
lr.fit(X_train_encoded[best_10], y_train)

y_pred_lr = lr.predict(X_valid_encoded[best_10])

In [232]:
gnb = GaussianNB()
gnb.fit(X_train_encoded[best_10], y_train)
y_pred_gnb = gnb.predict(X_valid_encoded[best_10])

In [233]:
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_encoded[best_10], y_train)
y_pred_knn = knn_model.predict(X_valid_encoded[best_10])

In [234]:
gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_lr
)
result_for_test["Lr_gini_my_valid"] = gini

gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_gnb
)
result_for_test["GNB_gini_my_valid"] = gini

gini = gini_fun(
    y_valid.to_numpy(),
    y_pred_knn
)
result_for_test["Knn_gini_my_valid"] = gini

In [235]:
for key, value in result_for_test.items():
  print(f"{key}: {value}")

Lr_gini_my_test: 0.2193608566029135
GNB_gini_my_test: 0.286839078154256
Knn_gini_my_test: 0.16150875133991915
Lr_gini_my_valid: 0.1986323888978756
GNB_gini_my_valid: 0.2678332191606527
Knn_gini_my_valid: 0.1649647888585941


Нет, явного падения качества между валидацией и тестом не наблюдается, Gini scores на тесте сопоставимы или даже немного лучше, чем на валидации.
Это говорит о том, что модель не переобучена и хорошо обобщается на новые данные, то есть переобучение отсутствует или минимально.

## 11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics.
Compare your algorithms on the test dataset using AUC PR metric.

In [236]:
def recall_my(y, y_pred, threshold = 0.5):
    y_pred_labels = (y_pred >= threshold).astype(int)
    TP = np.sum((y_pred_labels == 1) & (y == 1))
    FN = np.sum((y_pred_labels == 0) & (y == 1))
    result = TP / (TP + FN) if (TP + FN) > 0 else 0
    return result

print("Recall my:", recall_my(y_test, y_pred_lr))
print("Recall org:", recall_score(y_test, y_pred_lr))

Recall my: 0.044068919814446654
Recall org: 0.044068919814446654


In [237]:
def precision_my(y, y_pred, threshold = 0.5):
  y_pred_labels = (y_pred >= threshold).astype(int)
  TP = np.sum((y_pred_labels == 1) & (y == 1))
  FP = np.sum((y_pred_labels == 1) & (y == 0))
  result = TP / (TP + FP) if (TP + FP) > 0 else 0
  return result

print("Precision my:", precision_my(y_test, y_pred_lr))
print("Precision org:", precision_score(y_test, y_pred_lr))

Precision my: 0.11666666666666667
Precision org: 0.11666666666666667


In [238]:
def f1_my(y, y_pred, threshold = 0.5):
  y_pred_labels = (y_pred >= threshold).astype(int)
  TP = np.sum((y_pred_labels == 1) & (y == 1))
  FN = np.sum((y_pred_labels == 0) & (y == 1))
  FP = np.sum((y_pred_labels == 1) & (y == 0))
  result = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
  return result

print("F1 my:", f1_my(y_test, y_pred_lr))
print("F1 org:", f1_score(y_test, y_pred_lr))

F1 my: 0.06397306397306397
F1 org: 0.06397306397306397


In [239]:
def auc_pr(y, y_pred):
  thresholds = np.sort(np.unique(y_pred))
  precisions = []
  recalls = []

  for thr in thresholds:
    rec = recall_my(y, y_pred, threshold=thr)
    prec = precision_my(y, y_pred, threshold=thr)
    recalls.append(rec)
    precisions.append(prec)

  precisions = np.array(precisions + [1])
  recalls = np.array(recalls + [0])

  return precisions, recalls, thresholds

print("AUC PR my:", auc_pr (y_test, y_pred_lr))
print("AUC PR org:", precision_recall_curve(y_test, y_pred_lr))

AUC PR my: (array([0.12405459, 0.11666667, 1.        ]), array([1.        , 0.04406892, 0.        ]), array([0, 1]))
AUC PR org: (array([0.12405459, 0.11666667, 1.        ]), array([1.        , 0.04406892, 0.        ]), array([0, 1]))


## 12. Which hard label metric do you prefer for the task of detecting "lemon" cars?

Если важнее найти все "лимоны" и не пропустить ни одного, стоит ориентироваться на Recall. Если важнее не ошибаться в "лимонах", — Precision. Если баланс важен, то F1-score.